In [1]:
import numpy as np
import pandas as pd
import polars as pl
import time
import calendar
import calendar
import os

from datetime import datetime, timedelta
from pytrials.client import ClinicalTrials

In [4]:
ct = ClinicalTrials()
target_fields =["NCT Number", "Study Title", "Study Status", "Brief Summary", "Conditions", "Primary Outcome Measures",
                "Sponsor", "Collaborators", "Sex", "Age", "Enrollment", "Study Type", "Funder Type", "Start Date", "Completion Date"]

In [5]:
keywords = ["menstrual cycle","irregular menstruation","menorrhagia", "heavy menstrual bleeding", "abnormal uterine bleeding", "dysmenorrhea", "period cramps",
    "amenorrhea", "breakthrough bleeding","intermenstrual bleeding","combined oral contraceptives","progestin-only pill","IUD","intrauterine device","copper IUD",
    "levonorgestrel IUD","contraceptive side effects","birth control and mood","birth control and libido","tubal ligation","vulvovaginal candidiasis","recurrent yeast infection",
    "bacterial vaginosis","urinary tract infection","UTI","boric acid vaginal suppository","vaginal pH","vaginal microbiome","probiotics vaginal health","pelvic pain",
    "ovarian cyst","ovarian torsion","endometriosis","adenomyosis","pelvic floor dysfunction","pelvic floor physical therapy","vulvodynia","interstitial cystitis",
    "Bartholin cyst","breast lump","fibroadenoma","breast pain","mastalgia", "breast cancer screening","mammography","breast ultrasound","nipple pain","PCOS",
    "polycystic ovary syndrome","hormonal acne","hirsutism","thyroid dysfunction","hypothyroidism","Hashimoto's thyroiditis","perimenopause","menopause hormone therapy",
    "HRT","hot flashes","night sweats","sexually transmitted infection","STI","painful intercourse","dyspareunia","libido","sexual desire","unprotected sex",
    "pregnancy test","medical abortion","misoprostol","mifepristone","abortion access", "hair loss","alopecia","iron deficiency anemia","ferritin","bloating",
    "hemorrhoids","nausea","fatigue","sleep disturbance","heart palpitations","headache","migraine","allergic"]

In [7]:
output_data = "reddit_keywords_clinical_trials.csv"

for year in range(1986, 2027):
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        start_str3 = f"{year}-{month:02d}-01" 
        end_str3 = f"{year}-{month:02d}-{last_day:02d}"

        # Look through keywords list
        for keyword in keywords:
            keyword_month_query = f"{keyword} AND AREA[StartDate]RANGE[{start_str3}, {end_str3}]"

            try:
                fields3 = ct.get_study_fields(search_expr=keyword_month_query, fields=target_fields, max_studies=1000, fmt="csv")

                if fields3 and len(fields3) > 1:
                    df3 = pd.DataFrame.from_records(fields3[1:], columns = fields3[0])
                    df3['keyword'] = keyword
                    

                    file_name = os.path.isfile(output_data)
                    df3.to_csv(output_data, mode='a', header=not file_name, index=False)
                    print(f"Saved papers for '{keyword}' for this month {start_str3} - {end_str3}: {len(df3)}")
                    del df3  

                else:
                    print(f"No papers found for '{keyword}' for this month {start_str3} - {end_str3}")

            except Exception as e:
                print(f" Error for keyword '{keyword}' in {year}-{month:02d}: {e}")

            time.sleep(2)

No papers found for 'menstrual cycle' for this month 1986-01-01 - 1986-01-31
No papers found for 'irregular menstruation' for this month 1986-01-01 - 1986-01-31
No papers found for 'menorrhagia' for this month 1986-01-01 - 1986-01-31
No papers found for 'heavy menstrual bleeding' for this month 1986-01-01 - 1986-01-31
No papers found for 'abnormal uterine bleeding' for this month 1986-01-01 - 1986-01-31
No papers found for 'dysmenorrhea' for this month 1986-01-01 - 1986-01-31
No papers found for 'period cramps' for this month 1986-01-01 - 1986-01-31
No papers found for 'amenorrhea' for this month 1986-01-01 - 1986-01-31
No papers found for 'breakthrough bleeding' for this month 1986-01-01 - 1986-01-31
No papers found for 'intermenstrual bleeding' for this month 1986-01-01 - 1986-01-31
No papers found for 'combined oral contraceptives' for this month 1986-01-01 - 1986-01-31
No papers found for 'progestin-only pill' for this month 1986-01-01 - 1986-01-31
No papers found for 'IUD' for thi

## Load in both csvs
### Disclaimer: These two csvs will be deleted, but this is to keep track of the transformation of raw data to final csv

In [2]:
df = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/reddit_keywords_clinical_trials.csv")
df2 = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/reddit_keywords_clinical_trials._1956_1986.csv")

In [3]:
# Concat them together
dff = pd.concat([df, df2], ignore_index=True )

In [4]:
dff.info()

<class 'pandas.DataFrame'>
RangeIndex: 159936 entries, 0 to 159935
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   NCT Number                159936 non-null  str    
 1   Study Title               159936 non-null  str    
 2   Study Status              159936 non-null  str    
 3   Brief Summary             159936 non-null  str    
 4   Conditions                159936 non-null  str    
 5   Primary Outcome Measures  157441 non-null  str    
 6   Sponsor                   159936 non-null  str    
 7   Collaborators             56322 non-null   str    
 8   Sex                       159875 non-null  str    
 9   Age                       159936 non-null  str    
 10  Enrollment                159470 non-null  float64
 11  Funder Type               159936 non-null  str    
 12  Study Type                159936 non-null  str    
 13  Start Date                159936 non-null  str    
 14 

## Clean Data

### Duplicates

In [5]:
dff.info()

<class 'pandas.DataFrame'>
RangeIndex: 159936 entries, 0 to 159935
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   NCT Number                159936 non-null  str    
 1   Study Title               159936 non-null  str    
 2   Study Status              159936 non-null  str    
 3   Brief Summary             159936 non-null  str    
 4   Conditions                159936 non-null  str    
 5   Primary Outcome Measures  157441 non-null  str    
 6   Sponsor                   159936 non-null  str    
 7   Collaborators             56322 non-null   str    
 8   Sex                       159875 non-null  str    
 9   Age                       159936 non-null  str    
 10  Enrollment                159470 non-null  float64
 11  Funder Type               159936 non-null  str    
 12  Study Type                159936 non-null  str    
 13  Start Date                159936 non-null  str    
 14 

In [6]:
# Checking duplicates of nct number and study title
dff.duplicated(['NCT Number', 'Study Title']).sum()
# 58,394

np.int64(58394)

In [7]:
print(dff[['NCT Number', 'Study Title']].value_counts())

NCT Number   Study Title                                                                                               
NCT05837624  Estetrol/Drospirenone to Reduce the Average Size of Endometriomas                                             18
NCT07169292  Effects of Wet-Cupping on Polycystic Ovary Syndrome Patients                                                  18
NCT04256200  Efficacy of Dienogest Versus Oral Contraceptive Pills on Pain Associated With Endometriosis                   16
NCT07037082  Understanding Cycles to Improve Women's Health                                                                16
NCT00038467  Randomized Trial Of Exemestane Versus Continued Tamoxifen In Postmenopausal Women With Early Breast Cancer    15
                                                                                                                           ..
NCT00005283  Risk Factors For Asthma in Laboratory Animal Allergy                                                           

In [8]:
dff[dff['NCT Number'] == 'NCT05837624'].head(5)

,NCT Number,Study Title,Study Status,Brief Summary,Conditions,Primary Outcome Measures,Sponsor,Collaborators,Sex,Age,Enrollment,Funder Type,Study Type,Start Date,Completion Date,keyword
136712,NCT05837624,Estetrol/Drospirenone to Reduce the Average Si...,RECRUITING,"Endometriosis, a chronic gynecological disorde...",Ovarian Endometrioma|Endometrioma,Change in ovarian endometrioma volume at 6-mon...,Andrew Zakhari,NaN,FEMALE,"ADULT, OLDER_ADULT",21.0,OTHER,INTERVENTIONAL,2024-12-03,2026-12,dysmenorrhea
136718,NCT05837624,Estetrol/Drospirenone to Reduce the Average Si...,RECRUITING,"Endometriosis, a chronic gynecological disorde...",Ovarian Endometrioma|Endometrioma,Change in ovarian endometrioma volume at 6-mon...,Andrew Zakhari,NaN,FEMALE,"ADULT, OLDER_ADULT",21.0,OTHER,INTERVENTIONAL,2024-12-03,2026-12,period cramps
136724,NCT05837624,Estetrol/Drospirenone to Reduce the Average Si...,RECRUITING,"Endometriosis, a chronic gynecological disorde...",Ovarian Endometrioma|Endometrioma,Change in ovarian endometrioma volume at 6-mon...,Andrew Zakhari,NaN,FEMALE,"ADULT, OLDER_ADULT",21.0,OTHER,INTERVENTIONAL,2024-12-03,2026-12,amenorrhea
136726,NCT05837624,Estetrol/Drospirenone to Reduce the Average Si...,RECRUITING,"Endometriosis, a chronic gynecological disorde...",Ovarian Endometrioma|Endometrioma,Change in ovarian endometrioma volume at 6-mon...,Andrew Zakhari,NaN,FEMALE,"ADULT, OLDER_ADULT",21.0,OTHER,INTERVENTIONAL,2024-12-03,2026-12,breakthrough bleeding
136728,NCT05837624,Estetrol/Drospirenone to Reduce the Average Si...,RECRUITING,"Endometriosis, a chronic gynecological disorde...",Ovarian Endometrioma|Endometrioma,Change in ovarian endometrioma volume at 6-mon...,Andrew Zakhari,NaN,FEMALE,"ADULT, OLDER_ADULT",21.0,OTHER,INTERVENTIONAL,2024-12-03,2026-12,intermenstrual bleeding


In [10]:
dff.duplicated(['NCT Number', 'keyword']).sum()

np.int64(2)

In [11]:
# Remove duplicates
dfc = dff.drop_duplicates(subset = ['NCT Number', 'keyword'], keep='first')

In [ ]:
dfc.info()
# 159,993

<class 'pandas.DataFrame'>
RangeIndex: 159934 entries, 0 to 159933
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   NCT Number                159934 non-null  str    
 1   Study Title               159934 non-null  str    
 2   Study Status              159934 non-null  str    
 3   Brief Summary             159934 non-null  str    
 4   Conditions                159934 non-null  str    
 5   Primary Outcome Measures  157440 non-null  str    
 6   Sponsor                   159934 non-null  str    
 7   Collaborators             56321 non-null   str    
 8   Sex                       159873 non-null  str    
 9   Age                       159934 non-null  str    
 10  Enrollment                159469 non-null  float64
 11  Funder Type               159934 non-null  str    
 12  Study Type                159934 non-null  str    
 13  Start Date                159934 non-null  str    
 14 

### Rename columns

In [13]:
dfc = dfc.rename(columns={'NCT Number': 'nct_number',
                              'Study Title': 'title',
                              'Study Status': 'study_status',
                              'Brief Summary': 'summary',
                              'Conditions': 'conditions',
                              'Primary Outcome Measures': 'pom',
                              'Sponsor': 'sponsor',
                              'Collaborators': 'collaborators',
                              'Sex': 'sex',
                              'Age': 'age',
                              'Enrollment': 'enrollment',
                              'Funder Type': 'funder_type',
                              'Study Type': 'study_type',
                              'Start Date': 'start_date',
                              'Completion Date': 'completion_date'})

### Date string to date time

In [14]:
# This shows there is an inconsistent format between the different dates. We need to normalize them all to datetime - the time so the length will be 10
print(dfc['start_date'].astype(str).str.len().value_counts())
print(dfc['completion_date'].astype(str).str.len().value_counts())

start_date
10    109927
7      50007
Name: count, dtype: int64
completion_date
10.0    101500
7.0      56413
Name: count, dtype: int64


In [15]:
# Normalize dates to 10 
dfc['start_date'] = pd.to_datetime(dfc['start_date'], format='mixed', errors='coerce')
dfc['completion_date'] = pd.to_datetime(dfc['completion_date'], format='mixed', errors='coerce')

In [16]:
# Check
print(dfc['start_date'].astype(str).str.len().value_counts())
print(dfc['completion_date'].astype(str).str.len().value_counts())

start_date
10    159934
Name: count, dtype: int64
completion_date
10.0    157913
Name: count, dtype: int64


### Make years column

In [17]:
dfc['start_year'] = dfc['start_date'].dt.year
dfc['completion_year'] = dfc['completion_date'].dt.year
# Change completion year to int for some reason it is a float
dfc['completion_year'] = dfc['completion_year'].astype('Int64')

## DataFrame to CSV

In [19]:
dfc.to_csv("reddit_clinical_trials_data.csv", index=False)